# Notebook 2：模型推理与预测

**项目**：P2PNet-Soy 大豆种子定位与计数复现  
**论文**：Zhao et al., Plant Phenomics, 2022

本 Notebook 完成以下内容：
1. 从原始仓库克隆模型代码
2. 加载预训练权重（或训练模型）
3. 对测试集（B面）进行推理
4. 保存预测结果（图像 + CSV）

## 1. 克隆原始代码仓库

> 运行前请确保已安装 git

In [ ]:
import subprocess, sys, os
from pathlib import Path

# 克隆原始 P2PNet-Soy 仓库到 src/ 目录
src_dir = Path('../src')
p2p_dir = src_dir / 'P2PNet-Soy'

if not p2p_dir.exists():
    print('正在克隆 P2PNet-Soy 仓库...')
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/UTokyo-FieldPhenomics-Lab/P2PNet-Soy.git',
         str(p2p_dir)],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('克隆成功 ✓')
    else:
        print('克隆失败，请手动执行：')
        print(f'  git clone https://github.com/UTokyo-FieldPhenomics-Lab/P2PNet-Soy.git {p2p_dir}')
else:
    print(f'仓库已存在：{p2p_dir} ✓')

# 添加到 Python 路径
if str(p2p_dir) not in sys.path:
    sys.path.insert(0, str(p2p_dir))
print(f'已添加到 Python 路径：{p2p_dir}')

## 2. 环境准备

In [ ]:
import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备：{device}')
print(f'PyTorch 版本：{torch.__version__}')
print(f'CUDA 可用：{torch.cuda.is_available()}')

## 3. 数据路径配置

In [ ]:
# ===== 请根据实际路径修改 =====
DATA_ROOT   = Path('../data/Soybean_seed_counting')
MODEL_PATH  = Path('../src/weights/p2pnet_soy.pth')  # 模型权重路径
OUTPUT_DIR  = Path('../outputs')
# ==============================

IMG_B = DATA_ROOT / 'images_b'   # 测试集（B面）
LBL_B = DATA_ROOT / 'labels_b'

(OUTPUT_DIR / 'results').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)

print('路径配置完成 ✓')
print(f'测试图像目录：{IMG_B}')
print(f'模型权重路径：{MODEL_PATH}')

## 4. 加载模型

> 如果没有预训练权重，请先运行训练脚本。  
> 原始仓库训练命令：`python train.py --data_root <数据路径>`

In [ ]:
# 检查模型权重
if MODEL_PATH.exists():
    print(f'找到模型权重：{MODEL_PATH}')
    
    # 加载模型（根据原仓库的实际接口调整）
    # 参考原仓库 README 中的推理示例
    try:
        # 以下为示意代码，实际接口请参考原仓库
        checkpoint = torch.load(MODEL_PATH, map_location=device)
        # model = build_model(...)   # 根据原仓库接口
        # model.load_state_dict(checkpoint)
        # model.eval()
        print('模型加载成功 ✓')
    except Exception as e:
        print(f'加载失败：{e}')
        print('请参考原仓库 README 调整加载方式')
else:
    print('⚠️  未找到模型权重文件')
    print('请先训练模型，或从原作者处获取预训练权重')
    print('\n训练示例命令（参考原仓库）：')
    print('  cd src/P2PNet-Soy')
    print('  python train.py --data_root ../../data/Soybean_seed_counting')

## 5. 推理与预测

对测试集（B面）逐张进行推理，记录预测种子数量与真实数量。

In [ ]:
def load_gt_count(label_path):
    """读取真实标注点数量"""
    try:
        with open(label_path, 'r') as f:
            lines = [l.strip() for l in f if l.strip()]
        return len(lines)
    except:
        return -1

def preprocess_image(img_path):
    """图像预处理"""
    img = Image.open(img_path).convert('RGB')
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    return transform(img).unsqueeze(0).to(device)

# 收集测试集图像
test_imgs = sorted(list(IMG_B.glob('*.png')))
print(f'测试集图像数量：{len(test_imgs)}')

In [ ]:
# 推理循环（需要加载模型后取消注释）
results = []

for img_path in tqdm(test_imgs, desc='推理中'):
    lbl_path = LBL_B / (img_path.stem + '.txt')
    gt_count = load_gt_count(lbl_path)
    
    # ===== 此处替换为实际推理代码 =====
    # img_tensor = preprocess_image(img_path)
    # with torch.no_grad():
    #     outputs = model(img_tensor)
    # pred_count = len(outputs['pred_points'][0])  # 根据实际输出格式调整
    # ===================================
    
    # 占位符（替换为实际预测值）
    pred_count = -1  # TODO: 替换为实际推理结果
    
    results.append({
        'filename': img_path.stem,
        'gt_count': gt_count,
        'pred_count': pred_count
    })

df_results = pd.DataFrame(results)

# 保存结果
csv_path = OUTPUT_DIR / 'results' / 'predictions.csv'
df_results.to_csv(csv_path, index=False)
print(f'预测结果已保存至：{csv_path}')
print(df_results.head())

## 6. 推理结果可视化

展示预测点位叠加在原图上的效果（定性评估）。

In [ ]:
def visualize_prediction(img_path, gt_points, pred_points, ax):
    """可视化真实标注（绿色）与预测点（红色）"""
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    
    if gt_points:
        ax.scatter([p[0] for p in gt_points], [p[1] for p in gt_points],
                   c='lime', s=20, alpha=0.8, label=f'GT ({len(gt_points)})',
                   edgecolors='black', linewidths=0.5, zorder=5)
    if pred_points:
        ax.scatter([p[0] for p in pred_points], [p[1] for p in pred_points],
                   c='red', s=20, alpha=0.8, label=f'Pred ({len(pred_points)})',
                   marker='x', linewidths=1.5, zorder=6)
    
    ax.legend(fontsize=8, loc='upper right')
    ax.axis('off')
    ax.set_title(img_path.stem[:30], fontsize=8)

print('可视化函数已定义 ✓')
print('获得实际预测点位数据后，调用 visualize_prediction() 展示结果')

## 7. 下一步

完成推理后，请运行 `03_results_analysis.ipynb` 进行定量分析和最终报告生成。

**完成推理的检查清单：**
- [ ] 模型权重已加载
- [ ] 推理循环中已填入实际预测代码（替换占位符）
- [ ] `predictions.csv` 已成功生成
- [ ] 预测点位可视化效果合理